# Part 1: Interface Layer — Deploy CNN as FastAPI Service

## Objective
Learn how to wrap a trained CNN model in a production-ready REST API using FastAPI.

### What You Will Learn
- Wrap a PyTorch CNN in a FastAPI application
- Create a `/predict` endpoint that accepts image uploads
- Return structured JSON responses (prediction, confidence, model_version)
- Implement input validation with proper HTTP error codes
- Test the API using Swagger UI and structured test cases

### Error Handling Strategy
| Input Problem | HTTP Status | Meaning |
|---|---|---|
| Wrong file type (.txt, .pdf) | 400 Bad Request | Client sent invalid data |
| Corrupt image data | 400 Bad Request | Image cannot be decoded |
| Image too small (<32x32) | 400 Bad Request | Below minimum dimensions |
| Missing file parameter | 422 Unprocessable Entity | Required field missing |
| Internal model error | 500 Internal Server Error | Server-side failure |

---
## Step 1: Setup & Imports

In [1]:
import os
import io
import time
import logging
from pathlib import Path
from datetime import datetime
from typing import Dict, List

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models import resnet18

from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel

print(f"PyTorch: {torch.__version__}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

PyTorch: 2.1.0+cpu
Device: cpu


In [2]:
# Configuration
CLASS_NAMES = ['animal', 'name_board', 'pedestrian', 'pothole', 'road_sign', 'speed_breaker', 'vehicle']
NUM_CLASSES = len(CLASS_NAMES)
MODEL_PATH = './models/v1/model.pth'

# Image transformation pipeline (must match training)
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

print(f"Classes ({NUM_CLASSES}): {', '.join(CLASS_NAMES)}")

Classes (7): animal, name_board, pedestrian, pothole, road_sign, speed_breaker, vehicle


---
## Step 2: Define the CNN Model

In [3]:
class CNNModel(nn.Module):
    """ResNet-18 classifier for ADAS road hazard detection"""
    def __init__(self, num_classes):
        super().__init__()
        self.resnet = resnet18(weights=None)
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, num_classes)

    def forward(self, x):
        return self.resnet(x)


# Load model
model = CNNModel(NUM_CLASSES).to(DEVICE)

if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    print(f"Loaded trained weights from {MODEL_PATH}")
else:
    print(f"No weights found at {MODEL_PATH} — using random weights (demo mode)")

model.eval()
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

Loaded trained weights from ./models/v1/model.pth
Model parameters: 11,180,103


---
## Step 3: Build the FastAPI Application

We create three endpoints:
- `GET /health` — Is the server alive?
- `GET /info` — What model is loaded?
- `POST /predict` — Classify an uploaded image

In [4]:
# Response schemas
class PredictionResponse(BaseModel):
    prediction: str
    confidence: float
    class_probabilities: Dict[str, float]
    model_version: str
    latency_ms: float

class HealthResponse(BaseModel):
    status: str
    model_version: str
    device: str
    timestamp: str

class ModelInfoResponse(BaseModel):
    model_name: str
    version: str
    num_classes: int
    classes: List[str]
    input_shape: str
    device: str

print("Response schemas defined")

Response schemas defined


c:\Users\Lucifer\anaconda3\envs\venv_ai_quality\lib\site-packages\pydantic\_internal\_fields.py:149: UserWarning: Field "model_version" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
c:\Users\Lucifer\anaconda3\envs\venv_ai_quality\lib\site-packages\pydantic\_internal\_fields.py:149: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [ ]:
# Utility functions
def validate_image_file(file_ext: str) -> bool:
    """Validate file extension"""
    allowed = {'.jpg', '.jpeg', '.png', '.gif', '.bmp'}
    return file_ext.lower() in allowed

def process_image(contents: bytes) -> Image.Image:
    """Load and validate image from raw bytes"""
    try:
        return Image.open(io.BytesIO(contents)).convert('RGB')
    except Exception as e:
        raise ValueError(f"Could not open image: {str(e)}")

def validate_image_dimensions(image: Image.Image, min_size: int = 32) -> bool:
    """Check that image meets minimum size requirement"""
    return image.size[0] >= min_size and image.size[1] >= min_size

print("Utility functions defined")

In [ ]:
# Create FastAPI application
app = FastAPI(
    title="ADAS Classifier API",
    description="CNN model for detecting ADAS road hazards",
    version="1.0.0"
)


@app.get("/health", response_model=HealthResponse)
async def health_check():
    """API health check"""
    return HealthResponse(
        status="healthy",
        model_version="v1.0",
        device=str(DEVICE),
        timestamp=datetime.now().isoformat()
    )


@app.get("/info", response_model=ModelInfoResponse)
async def model_info():
    """Get model metadata"""
    return ModelInfoResponse(
        model_name="ResNet-18 ADAS Detector",
        version="1.0",
        num_classes=NUM_CLASSES,
        classes=CLASS_NAMES,
        input_shape="128x128x3",
        device=str(DEVICE)
    )


@app.post("/predict", response_model=PredictionResponse)
async def predict(file: UploadFile = File(...)):
    """
    Classify an uploaded image.
    Returns prediction, confidence, and per-class probabilities.
    """
    start_time = time.time()

    try:
        # 1. Validate file type
        file_ext = Path(file.filename).suffix.lower()
        if not validate_image_file(file_ext):
            raise HTTPException(
                status_code=400,
                detail=f"Invalid file type: {file_ext}. Allowed: jpg, jpeg, png, gif, bmp"
            )

        # 2. Read file bytes
        contents = await file.read()
        if not contents or len(contents) == 0:
            raise HTTPException(status_code=400, detail="Empty file - no data received")

        # 3. Decode image
        try:
            image = process_image(contents)
        except ValueError as e:
            raise HTTPException(status_code=400, detail=str(e))

        # 4. Validate dimensions
        if not validate_image_dimensions(image):
            raise HTTPException(
                status_code=400,
                detail=f"Image too small: {image.size}. Minimum: 32x32"
            )

        # 5. Transform and predict
        image_tensor = transform(image).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            outputs = model(image_tensor)
            probabilities = torch.softmax(outputs, dim=1)
            confidence, predicted_idx = torch.max(probabilities, 1)

        predicted_class = CLASS_NAMES[predicted_idx.item()]
        class_probs = {
            name: float(probabilities[0, i].item())
            for i, name in enumerate(CLASS_NAMES)
        }
        latency = (time.time() - start_time) * 1000

        return PredictionResponse(
            prediction=predicted_class,
            confidence=round(confidence.item(), 4),
            class_probabilities=class_probs,
            model_version="v1.0",
            latency_ms=round(latency, 2)
        )

    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


print("FastAPI app created with 3 endpoints: /health, /info, /predict")
print("In production, run: uvicorn api_simple:app --reload --port 8000")
print("Then open http://localhost:8000/docs for Swagger UI")

---
## Step 4: Test the API with Structured Test Cases

We use FastAPI's `TestClient` to test without starting a server.

### Test Plan
| # | Test Case | Input | Expected Status | Validates |
|---|-----------|-------|----------------|-----------|
| 1 | Health check | GET /health | 200 | Server is alive |
| 2 | Model info | GET /info | 200 | Metadata correct |
| 3 | Valid PNG | 100x100 red PNG | 200 | Happy path |
| 4 | Valid JPEG | 100x100 green JPEG | 200 | Format support |
| 5 | Large image | 1024x1024 PNG | 200 | Resize handling |
| 6 | Invalid file type | .txt file | 400 | File type check |
| 7 | Corrupt image | Random bytes as .png | 400 | Decode check |
| 8 | Empty file | 0 bytes | 400 | Empty check |
| 9 | Tiny image | 16x16 PNG | 400 | Dimension check |
| 10 | Missing file | No file in POST | 422 | Required param |

In [ ]:
# Create test client
test_client = TestClient(app)

# Helper to create test images
def create_test_image(width=100, height=100, color=(255, 0, 0), fmt="PNG"):
    img = Image.new("RGB", (width, height), color=color)
    buf = io.BytesIO()
    img.save(buf, format=fmt)
    buf.seek(0)
    return buf

print("TestClient ready")

In [ ]:
# Run all test cases
test_results = []

# Test 1: Health check
r = test_client.get("/health")
passed = r.status_code == 200 and r.json()["status"] == "healthy"
test_results.append(("Health check", 200, r.status_code, passed))

# Test 2: Model info
r = test_client.get("/info")
passed = r.status_code == 200 and r.json()["num_classes"] == 7
test_results.append(("Model info", 200, r.status_code, passed))

# Test 3: Valid PNG
img = create_test_image(100, 100, (255, 0, 0), "PNG")
r = test_client.post("/predict", files={"file": ("test.png", img, "image/png")})
passed = r.status_code == 200 and "prediction" in r.json()
test_results.append(("Valid PNG", 200, r.status_code, passed))

# Test 4: Valid JPEG
img = create_test_image(100, 100, (0, 255, 0), "JPEG")
r = test_client.post("/predict", files={"file": ("test.jpg", img, "image/jpeg")})
passed = r.status_code == 200
test_results.append(("Valid JPEG", 200, r.status_code, passed))

# Test 5: Large image
img = create_test_image(1024, 1024)
r = test_client.post("/predict", files={"file": ("large.png", img, "image/png")})
passed = r.status_code == 200
test_results.append(("Large image (1024x1024)", 200, r.status_code, passed))

# Test 6: Invalid file type
buf = io.BytesIO(b"hello world")
r = test_client.post("/predict", files={"file": ("test.txt", buf, "text/plain")})
passed = r.status_code == 400
test_results.append(("Invalid file type (.txt)", 400, r.status_code, passed))

# Test 7: Corrupt image
buf = io.BytesIO(b"not a real image file at all")
r = test_client.post("/predict", files={"file": ("corrupt.png", buf, "image/png")})
passed = r.status_code == 400
test_results.append(("Corrupt image data", 400, r.status_code, passed))

# Test 8: Empty file
buf = io.BytesIO(b"")
r = test_client.post("/predict", files={"file": ("empty.png", buf, "image/png")})
passed = r.status_code == 400
test_results.append(("Empty file", 400, r.status_code, passed))

# Test 9: Tiny image
img = create_test_image(16, 16)
r = test_client.post("/predict", files={"file": ("tiny.png", img, "image/png")})
passed = r.status_code == 400
test_results.append(("Tiny image (16x16)", 400, r.status_code, passed))

# Test 10: Missing file
r = test_client.post("/predict")
passed = r.status_code == 422
test_results.append(("Missing file parameter", 422, r.status_code, passed))

# Print results
print(f"{'Test Case':<30} {'Expected':>8} {'Got':>8} {'Result':>8}")
print("-" * 60)
for name, expected, got, passed in test_results:
    status = "PASS" if passed else "FAIL"
    print(f"{name:<30} {expected:>8} {got:>8} {status:>8}")

passed_count = sum(1 for _, _, _, p in test_results if p)
print(f"\nResults: {passed_count}/{len(test_results)} tests passed")

---
## Step 5: Examine a Successful Prediction

In [ ]:
# Make a prediction and inspect the full response
img = create_test_image(128, 128, (100, 150, 200))
response = test_client.post("/predict", files={"file": ("sample.png", img, "image/png")})
data = response.json()

print("Response fields:")
print(f"  prediction:         {data['prediction']}")
print(f"  confidence:         {data['confidence']}")
print(f"  model_version:      {data['model_version']}")
print(f"  latency_ms:         {data['latency_ms']}")
print(f"\nClass probabilities:")
for cls, prob in sorted(data['class_probabilities'].items(), key=lambda x: -x[1]):
    bar = '#' * int(prob * 50)
    print(f"  {cls:<15} {prob:.4f} {bar}")

print(f"\nProbabilities sum: {sum(data['class_probabilities'].values()):.4f}")

---
## Step 6: Visualize Results

In [ ]:
# Measure latency across multiple requests
latencies = []
for i in range(20):
    img = create_test_image(128, 128, (i * 12, i * 8, i * 5))
    start = time.time()
    r = test_client.post("/predict", files={"file": (f"bench_{i}.png", img, "image/png")})
    elapsed = (time.time() - start) * 1000
    if r.status_code == 200:
        latencies.append(elapsed)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Test results bar chart
names = [r[0] for r in test_results]
colors = ['green' if r[3] else 'red' for r in test_results]
axes[0].barh(range(len(names)), [1]*len(names), color=colors)
axes[0].set_yticks(range(len(names)))
axes[0].set_yticklabels(names, fontsize=9)
axes[0].set_title('Test Results')
axes[0].set_xlabel('Pass (green) / Fail (red)')
axes[0].invert_yaxis()

# Latency histogram
axes[1].hist(latencies, bins=10, color='steelblue', edgecolor='black', alpha=0.7)
axes[1].axvline(np.mean(latencies), color='red', linestyle='--', label=f'Mean: {np.mean(latencies):.1f}ms')
axes[1].set_title('Prediction Latency Distribution')
axes[1].set_xlabel('Latency (ms)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Step 7: Summary Report

In [ ]:
print("=" * 60)
print("INTERFACE LAYER TEST REPORT")
print("=" * 60)

print(f"\nTest Results: {passed_count}/{len(test_results)} passed")
for name, expected, got, passed in test_results:
    icon = "[PASS]" if passed else "[FAIL]"
    print(f"  {icon} {name}")

print(f"\nLatency Statistics (n={len(latencies)}):")
print(f"  Mean:   {np.mean(latencies):.2f} ms")
print(f"  Median: {np.median(latencies):.2f} ms")
print(f"  Min:    {np.min(latencies):.2f} ms")
print(f"  Max:    {np.max(latencies):.2f} ms")
print(f"  Std:    {np.std(latencies):.2f} ms")

print(f"\nEndpoints Implemented:")
print(f"  GET  /health  — Health check")
print(f"  GET  /info    — Model information")
print(f"  POST /predict — Image classification")
print(f"\nValidation Checks:")
print(f"  File type validation    -> 400")
print(f"  Corrupt image detection -> 400")
print(f"  Empty file detection    -> 400")
print(f"  Minimum size check      -> 400")
print(f"  Missing parameter       -> 422")
print(f"  Internal error          -> 500")
print("=" * 60)

---
## Key Takeaways

1. **FastAPI auto-generates documentation**: Visit `/docs` for interactive Swagger UI that lets you test endpoints directly in the browser.

2. **Input validation prevents crashes**: Every user input (file type, image data, dimensions) is validated before reaching the model. This prevents cryptic model errors and provides clear feedback.

3. **HTTP status codes communicate error types**:
   - `200` = Success
   - `400` = Client sent bad data (wrong format, corrupt file)
   - `422` = Client missed a required field
   - `500` = Something broke on the server

4. **TestClient enables testing without a running server**: FastAPI's TestClient simulates HTTP requests in-process, making tests fast and deterministic.

5. **Structured responses** (Pydantic models) ensure the API always returns consistent JSON, making it easier for downstream consumers to parse.

### Running the Standalone API
```bash
# Start the API server
uvicorn api_simple:app --reload --port 8000

# Open Swagger UI
# http://localhost:8000/docs

# Run tests
pytest test_part1.py -v
```